# 🏋️ DeepSeek-V4 训练 — Miles + Megatron-LM 六维并行

**本文目标**：深入理解 DeepSeek-V4 的训练基础设施——六种并行策略、RL 训练流水线和数值稳定性保障。

读完这篇你会理解：
- Miles 框架如何整合 Megatron-LM 的六种并行
- mHC 如何影响流水线并行的通信模式
- FP8 Rollout + BF16 Training 的混合精度方案
- 训练中的数值稳定性挑战和解决方案

## 1. 六种并行策略全景

### 1.1 Megatron-LM 六维并行

```
DeepSeek-V4 训练使用 Miles (DeepSeek 自研框架) + Megatron-LM

并行策略: DP + TP + SP + EP + PP + CP

┌─────────────────────────────────────────────────────────────┐
│                      六维并行全景                             │
├──────────────┬──────────────────────────────────────────────┤
│ DP (Data)    │ 数据并行: 每个 GPU 处理不同的 batch             │
│              │ ZeRO-3: 分片优化器状态 + 梯度 + 参数           │
├──────────────┼──────────────────────────────────────────────┤
│ TP (Tensor)  │ 张量并行: 单个层的权重切分到多个 GPU            │
│              │ 用于 Attention (QKV proj) 和 FFN 的大矩阵      │
├──────────────┼──────────────────────────────────────────────┤
│ SP (Sequence)│ 序列并行: 长序列切分到多个 GPU                  │
│              │ 用于 LayerNorm/Dropout (不切分权重的部分)      │
├──────────────┼──────────────────────────────────────────────┤
│ EP (Expert)  │ 专家并行: MoE 专家分布到不同 GPU               │
│              │ DeepEP: all-to-all 通信分发 tokens            │
├──────────────┼──────────────────────────────────────────────┤
│ PP (Pipeline)│ 流水线并行: 不同层分布到不同 GPU                │
│              │ mHC 的 4 流需要跨 stage 传递                   │
├──────────────┼──────────────────────────────────────────────┤
│ CP (Context) │ 上下文并行: Attention 的序列维度切分            │
│              │ 仅用于 prefill 的 NSA(非自回归)路径             │
└──────────────┴──────────────────────────────────────────────┘
```

### 1.2 mHC 对 PP 的影响

```
标准 Transformer + PP:
  层间传递: [seq, batch, hidden] 三维张量
  p2p 通信: send(activation) → 下游 GPU recv

V4 + mHC + PP:
  mHC 有 hc_mult 个并行分支 → 4 条"流"
  层间传递: [seq, batch, hc_mult=4, hidden] 四维张量!
  
  → 需要特殊的 p2p 通信: 发送整个四维张量
  → 下游 GPU 接收后按 hc_mult 维度拆分到 4 个流
  → 然后做 Sinkhorn 归一化得到混合权重

开销:
  四维 vs 三维: 数据量 4x (但 hc_mult=4, 实际 overhead 不大)
  通信模式: 与标准 PP 类似 (只是多了第 4 维)
```

### 1.3 CP (上下文并行) 在 C4/C128 的特殊处理

```
C4 层 + CP:
  C4 压缩器的 overlap 变换跨 CP rank 边界
  → 每个 rank 的局部压缩窗口与相邻 rank 有重叠
  
  解决方案: 用一个 all-gather 同时解决:
    1. halo exchange (边界数据交换)
    2. indexer 需求 (为 top-k 提供全局视图)
  
  → 一个通信操作完成两种需求 → 减少通信轮次

C128 层 + CP:
  C128 无 overlap 变换 (每个 rank 独立压缩)
  → 跳过 all-gather → 更简单
```

## 2. 训练精度方案

### 2.1 权重精度: FP32 主权重 + FP8 前向

```
训练精度策略:
  主权重: FP32 (存储在 CPU, 通过 ZeRO-3 分片)
  前向传播: FP8 (减少显存, 加速计算)
  反向传播: BF16 (梯度累加需要更多精度)
  优化器状态: FP32 (Adam 的 m/v 需要高精度)
  
  专家权重: 训练时 BF16, 推理时 → FP4 (后训练量化?)
  
敏感路径保护:
  某些权重和梯度对精度敏感:
  - mHC 的 Sinkhorn 路径: FP32
  - MoE router gate: FP32
  - Per-expert score bias: FP32
  - Hash routing 的早期层: FP32
  
  这些路径在训练时"卡住" (冻结某些参数)
  或被强制使用 FP32 精度
```

### 2.2 FP8 Rollout + BF16/FP8 Training

```
RL 训练的混合精度方案:

Rollout (生成):
  使用 FP8 权重和 FP8 激活
  → 与推理时的精度匹配 (SGLang kernel-level numerical match)
  → 通过 Attention QAT (Quantization-Aware Training):
      模拟 FP8 激活量化 → 使 rollout 的数值行为与推理引擎一致
      
Training (更新):
  使用 BF16 或 FP8
  → Rollout Routing Replay (R3): 复现 rollout 时的路由决策
  → 扩展到 (b, s, h, d) 格式 (由 V3 的 (b*s, h, d) 升级)
  
数值漂移:
  首步 rollout/training log-prob 漂移 ≈ 0.023
  → 在可接受范围内
  → 随训练步骤增加, 漂移保持稳定 (不随时间发散)
```

### 2.3 Indexer Replay (实验性)

```
Indexer Replay 是一个实验性功能:

Rollout 阶段:
  记录每层 C4 indexer 的 top-k 选择 (哪些压缩位置被选中)

Training 阶段:
  通过 rollout 通道传回训练侧
  在每 C4 层重新注入这些选择
  → 训练时使用与 rollout 完全一致的稀疏模式
  → 减少 train-inference 的分布偏移

这是类似于 "Rollout Routing Replay" 的思想
  但应用于 C4 的 indexer 选择而非 MoE routing
```

## 3. 数值稳定性保障

### 3.1 为什么需要特殊处理？

```
V4 的架构创新带来新的数值挑战:

1. mHC 的 Sinkhorn 归一化:
   → 需要连续做行/列归一化 → 容易累积误差
   
2. C4/C128 压缩器的反向传播:
   → 对长序列做 softmax → 沿长轴求和 → BF16 精度不够
   → 会出现 bias (累加方向性误差)
   
3. 混合 FP32/BF16 优化器:
   → 不同参数组不同精度 → checkpoint 保存/恢复时需要特殊处理
```

### 3.2 解决方案清单

```
 1. 压缩器 backward all-reduce → 切换为 FP32
    防止 BF16 沿长轴求和累积偏置

 2. 选择性冻结不稳定路径:
    - mHC Sinkhorn
    - MoE router gate
    - per-expert score bias
    - hash routing 早期层
    (这些路径的权重不参与训练, 或使用 FP32)

 3. 确定性操作设置:
    - cuDNN deterministic mode
    - NCCL Ring (而非 Tree, 保证通信确定性)
    - 关闭 TransformerEngine 的非确定性路径
    - cuBLAS 固定 workspace
    → 代价: ~10-15% 吞吐下降
    → 收益: 训练可复现 + 避免非确定性导致的数值发散

 4. Checkpoint 精度修复:
    - 分布式优化器在混合 FP32/BF16 组下
    - 保存/恢复时需要正确处理精度转换
    - 恢复时将优化器状态分配到 CPU (避免 GPU OOM)
```

## 4. RL 训练流水线

```
RL 训练 (DAPO/GRPO 类方法):

  Step 1: Rollout (FP8)
    模型用 FP8 权重生成 responses (max 4096 tokens)
    R3 捕获 routing 决策, Indexer Replay 捕获 indexer 选择
    
  Step 2: Reward Calculation
    基于 ground truth / verifier 计算奖励
    
  Step 3: Advantage Estimation
    GRPO: group-based advantage
    
  Step 4: Policy Update (BF16/FP8)
    使用收集的 rollout 数据更新策略
    → 奖励和评估分数随训练稳定增长

训练配置 (285B 模型, DAPO):
  Hardware: 32 × GB300 GPU
  Parallel: TP + SP + EP + PP
  Response length: 4096 (max)
  Rollout: FP8, Training: BF16
  R3: enabled
```